<a href="https://colab.research.google.com/github/gautamstrike789/MachineLearningPractice/blob/main/exercises/machine-learning/supervised-learning/ensemble_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Learning
You should build a machine learning pipeline using an ensemble learning model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Conduct data exploration, data preprocessing, and feature engineering if necessary.
- Train and test an ensemble learning model, such as [random forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) or [gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html).
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

# Importing Libraries


In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix


# Loading Dataset


In [9]:
df=pd.read_csv("/content/mnist.csv")
df

,id,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,31953,5,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,34452,8,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,60897,5,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,36953,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1981,3,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2280,59842,7,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2281,43334,8,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2282,51104,8,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2283,61031,2,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
df = df.set_index('id')

In [11]:
x= df.drop("class", axis=1)
y= df["class"]

# Splitting data

In [12]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

print("df:", df.shape)
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

df: (2285, 785)
x_train: (1828, 784)
x_test: (457, 784)
y_train: (1828,)
y_test: (457,)


# Algorithm selection and Hyperparameter Tuning

# Random Forest

In [13]:
parameter_grid = {
    "criterion": ["gini", "entropy"],
    "n_estimators": range(50,260,50), #[50, 100, 150, 200, 250]
}
model_1 = GridSearchCV(RandomForestClassifier(), parameter_grid, cv=5, scoring="accuracy", n_jobs=-1)
model_1.fit(x_train, y_train)
print("Accuracy of best Random Forest classifier:", model_1.best_score_)
print("Best found hyperparameter:", model_1.best_params_)

Accuracy of best Random Forest classifier: 0.9239658657085112
Best found hyperparameter: {'criterion': 'entropy', 'n_estimators': 200}


# Gradient Boosting

In [17]:
parameter_grid= {
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": range(50,260,50),
}

# Handle NaN values: GradientBoostingClassifier does not accept NaN.
# Fill NaN with 0, assuming missing pixel values are black.
x_train = x_train.fillna(0)
x_test = x_test.fillna(0) # Also process test set for consistency

model_2 = GridSearchCV(GradientBoostingClassifier(), parameter_grid, cv=5, scoring="accuracy", n_jobs=-1)
# Use the processed training data
model_2.fit(x_train, y_train)
print("Accuracy of best Gradient Boosting classifier:", model_2.best_score_)
print("Best found hyperparameter:", model_2.best_params_)

Accuracy of best Gradient Boosting classifier: 0.9009940863837113
Best found hyperparameter: {'learning_rate': 0.2, 'n_estimators': 250}


# Testing the Best Model

In [22]:
from sklearn.metrics import precision_recall_fscore_support

y_predicted= model_1.predict(x_test)
accuracy= accuracy_score(y_test, y_predicted)
cm= confusion_matrix(y_test, y_predicted)
precision, recall, f1, support= precision_recall_fscore_support(y_test, y_predicted)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Confusion Matrix:\n", cm)

Accuracy: 0.9102844638949672
Precision: [0.95744681 0.9        0.86842105 0.83928571 0.89189189 0.87804878
 0.94117647 0.93478261 0.93103448 0.975     ]
Recall: [0.97826087 0.96428571 0.94285714 0.90384615 0.91666667 0.7826087
 0.91428571 0.93478261 0.8852459  0.88636364]
F1-score: [0.96774194 0.93103448 0.90410959 0.87037037 0.90410959 0.82758621
 0.92753623 0.93478261 0.90756303 0.92857143]
Confusion Matrix:
 [[45  0  0  0  0  0  0  0  1  0]
 [ 0 54  0  1  0  1  0  0  0  0]
 [ 0  1 33  0  0  0  0  0  1  0]
 [ 0  0  2 47  0  2  0  1  0  0]
 [ 1  0  0  0 33  0  1  0  0  1]
 [ 1  1  0  5  0 36  1  0  2  0]
 [ 0  1  1  0  0  1 32  0  0  0]
 [ 0  1  1  0  1  0  0 43  0  0]
 [ 0  2  1  3  0  1  0  0 54  0]
 [ 0  0  0  0  3  0  0  2  0 39]]
